In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
from rich import print
from loguru import logger

# グローバル共有状態を定義
class OverAllState(TypedDict):
    raw_text: str # 未加工のテキスト
    cleaned_text: str # クリーニング後のテキスト

# サブグラフを構築
def subgraph_strip_node(state: OverAllState) -> OverAllState:
    logger.info("サブグラフの subgraph_strip_node ノードが実行されました")
    raw_text = state["raw_text"]
    stripped_text = raw_text.strip()

    return {
        "cleaned_text": stripped_text
    }

def subgraph_punctuate_node(state: OverAllState) -> OverAllState:
    logger.info("サブグラフの subgraph_punctuate_node ノードが実行されました")
    cleaned_text = state["cleaned_text"]
    punctuation = interrupt("文末に追加したい句読点は？[。/、/！/？/；]")
    punctuation1 = interrupt("文末に追加したい句読点は？[。/、/！/？/；]")
    punctuated_text = cleaned_text + punctuation + punctuation1

    return {
        "cleaned_text": punctuated_text
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("subgraph_strip_node", subgraph_strip_node)
builder.add_node("subgraph_punctuate_node", subgraph_punctuate_node)

builder.add_edge(START, "subgraph_strip_node")
builder.add_edge("subgraph_strip_node", "subgraph_punctuate_node")
builder.add_edge("subgraph_punctuate_node", END)

# ==================================================
# 【戦略切り替えポイント】この行だけを変更すればよい
subgraph = builder.compile()  # Per-invocation（默认）
# subgraph = builder.compile(checkpointer=True)   # Per-thread
# subgraph = builder.compile(checkpointer=False)  # Stateless（中断不可）
# ==================================================

# 親グラフを構築
builder = StateGraph(state_schema=OverAllState)
builder.add_node("subgraph_node", subgraph)

builder.add_edge(START, "subgraph_node")
builder.add_edge("subgraph_node", END)

checkpointer = InMemorySaver()
parent_graph = builder.compile(checkpointer=checkpointer)

raw_text = "   LangGraph は本当に面白い        "
config = {"configurable": {"thread_id": "123"}}
# 初回呼び出し
interrupted_res = parent_graph.invoke(
    {"raw_text": raw_text},
    config=config
)
print("=" * 30, "->   中断情報   <-", "=" * 30)
print(interrupted_res)


from IPython.display import display, Image
display(
    Image(
        parent_graph
        .get_graph(xray=True)
        .draw_mermaid_png()
    )
)


In [ ]:

# 再開呼び出し
parent_graph.invoke(
    Command(resume="！"),
    config=config
)
res = parent_graph.invoke(
    Command(resume="！"),
    config=config
)

cleaned_text = res["cleaned_text"]
print("=" * 30, "->   元のテキスト   <-", "=" * 30)
print(raw_text)
print("=" * 30, "-> クリーニング後のテキスト <-", "=" * 30)
print(cleaned_text)